# 🏥💰 Financial Triage — Train an AI Financial Advisor for India

**What this notebook does:**
1. Installs Unsloth + TRL
2. Clones the environment from HuggingFace (stochastic bills ±15%, partial observability)
3. SFT warmup using 180 expert examples (JSONL)
4. GRPO training against the LIVE environment (actions executed in real env)
5. Generates before/after reward plots

**Runtime:** GPU → T4 optional for 3B, A100 / large GPU for **7B** (figures in README)

**No API key needed.** Model weights are free from HuggingFace.

**To switch between 3B and 7B:** Change ONE line in Cell 4 (`model_name = ...`).

**Experiment tracking (recommended for hackathon runs):** set `WANDB_API_KEY` in Colab **Secrets** (or `export` locally). SFT and GRPO use `report_to='wandb'` when the key is present; otherwise they log to console only. After a run, add your public W&B run URL to the project README for judges.

## Cell 1: Install Dependencies (restart runtime after this)

In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.12.0" "transformers>=4.46.0" "datasets>=3.0.0"
!pip install -q "openenv-core[core]>=0.2.3" "pydantic>=2.0.0" "matplotlib"
!pip install -q "accelerate>=0.34.0" "bitsandbytes>=0.44.0"
!pip install -q "openai>=1.0.0" "uvicorn>=0.27.0" "fastapi>=0.109.0"
print('✅ All dependencies installed. RESTART RUNTIME NOW, then run Cell 2.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ All dependencies installed. RESTART RUNTIME NOW, then run Cell 2.


## Cell 2: Clone Environment + Verify

In [2]:
import os, sys
if not os.path.exists('financial-triage-env'):
    !git clone https://huggingface.co/spaces/indra-dhanush/financial-triage-env
sys.path.insert(0, 'financial-triage-env')

from server.my_env_environment import MyEnvironment
from models import FinancialAction, ActionType
from inference import _heuristic_action, observation_to_prompt, parse_action, SYSTEM_PROMPT

# Smoke test
env = MyEnvironment()
obs = env.reset(seed=42, task_id='easy')
print(f'✅ Environment loaded! Easy task: {obs.episode_length} days, checking=INR {obs.account.checking_balance:,.0f}')

Cloning into 'financial-triage-env'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 143 (delta 21), reused 0 (delta 0), pack-reused 83 (from 1)
Receiving objects: 100% (143/143), 597.21 KiB | 2.91 MiB/s, done.
Resolving deltas: 100% (57/57), done.
✅ Environment loaded! Easy task: 30 days, checking=INR 50,000


## Cell 3: Run Baseline (Heuristic Agent) — Before Training

In [4]:
baseline_scores = {}
for task_id in ['easy', 'medium', 'hard']:
    scores = []
    for seed in range(5):
        env = MyEnvironment()
        obs = env.reset(seed=seed, task_id=task_id)
        for _ in range(obs.episode_length):
            obs = env.step(_heuristic_action(obs))
        scores.append(env.get_episode_score())
    baseline_scores[task_id] = scores
    print(f'  {task_id:>6} baseline: {sum(scores)/len(scores):.4f}')
print('✅ Baseline recorded')

    easy baseline: 0.9990
  medium baseline: 0.6937
    hard baseline: 0.4266
✅ Baseline recorded


## Cell 4: Load Model with Unsloth (4-bit, no API key needed)

In [5]:
from unsloth import FastLanguageModel
import torch

# === CHOOSE YOUR MODEL SIZE (change this ONE line for 8B) ===
#model_name = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'   # T4 (free Colab)
model_name = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' # A100 (HF credits)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name, max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
)
print(f'✅ Model loaded: {model_name}')
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Model loaded: unsloth/Qwen2.5-7B-Instruct-bnb-4bit
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## Cell 5: SFT Warmup — Teach the Model Valid Actions (180 examples)

In [10]:
import gc, torch, os
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

gc.collect()
torch.cuda.empty_cache()

dataset = load_dataset('json', data_files='financial-triage-env/sft_dataset.jsonl', split='train')
print(f'SFT dataset: {len(dataset)} examples')

# Pre-convert messages → plain text (avoids formatting_func issues)
def convert_to_text(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(convert_to_text, remove_columns=dataset.column_names)
print(f'Sample: {dataset[0]["text"][:200]}...')

model.gradient_checkpointing_enable()

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=1024,
    dataset_num_proc=1,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs/sft',
        report_to='wandb' if os.environ.get('WANDB_API_KEY') else 'none',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
)

print('🚀 SFT warmup starting...')
sft_stats = trainer.train()
model.save_pretrained('outputs/sft')
tokenizer.save_pretrained('outputs/sft')
print(f'✅ SFT done! Loss: {sft_stats.training_loss:.4f}')

del trainer
gc.collect()
torch.cuda.empty_cache()


SFT dataset: 180 examples


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Sample: <|im_start|>system
You are an AI financial advisor managing an Indian household's finances day by day.
All amounts are in Indian Rupees (INR). Salary is credited via UPI.
Each day you must choose EXAC...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/180 [00:00<?, ? examples/s]

🚀 SFT warmup starting...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 180 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.245723
2,2.248597
3,2.143156
4,2.018422
5,1.935905
6,1.767952
7,1.736520
8,1.569439
9,1.468375
10,1.394881


Unsloth: Restored added_tokens_decoder metadata in outputs/sft/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/sft/tokenizer_config.json.


✅ SFT done! Loss: 0.5317


## Cell 6: Evaluate SFT Model Against Live Environment

In [12]:
FastLanguageModel.for_inference(model)

def run_trained_episode(model, tokenizer, task_id, seed=42):
    env = MyEnvironment()
    obs = env.reset(seed=seed, task_id=task_id)
    for day in range(obs.episode_length):
        obs_text = observation_to_prompt(obs)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': obs_text},
        ]
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip().split('\n')[0]
        action = parse_action(response, obs)
        if action is None:
            action = _heuristic_action(obs)
        obs = env.step(action)
    return env.get_episode_score()

sft_scores = {}
for task_id in ['easy', 'medium', 'hard']:
    scores = [run_trained_episode(model, tokenizer, task_id, seed=s) for s in range(5)]
    sft_scores[task_id] = scores
    print(f'  {task_id:>6} SFT: {sum(scores)/len(scores):.4f}  (baseline: {sum(baseline_scores[task_id])/len(baseline_scores[task_id]):.4f})')
print('✅ SFT evaluation done')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_m

    easy SFT: 0.9327  (baseline: 0.9990)


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  medium SFT: 0.5960  (baseline: 0.6937)


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

    hard SFT: 0.4054  (baseline: 0.4266)
✅ SFT evaluation done


## Cell 7: GRPO Training — RL Against Live Environment

Each action the model proposes is **executed in the actual environment** and scored.

In [ ]:
import os
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

FastLanguageModel.for_training(model)

# Build prompts from environment states (multiple seeds for stochastic diversity)
prompts = []
for task_id in ['easy', 'medium', 'hard']:
    for seed in range(3):
        env = MyEnvironment()
        obs = env.reset(seed=seed, task_id=task_id)
        for day in range(min(obs.episode_length, 20)):
            obs_text = observation_to_prompt(obs)
            prompts.append({
                'prompt': [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':obs_text}],
                'task_id': task_id, 'seed': seed, 'day': day,
            })
            obs = env.step(_heuristic_action(obs))

grpo_dataset = Dataset.from_list(prompts)
print(f'GRPO dataset: {len(grpo_dataset)} prompts')

def reward_fn(prompts, completions, **kwargs):
    """Reward = run the action in a LIVE environment and score it."""
    rewards = []
    for completion in completions:
        # completions can be list of dicts or strings
        if isinstance(completion, list):
            action_text = completion[-1]['content'] if completion else ''
        else:
            action_text = str(completion)
        action_text = action_text.strip().split('\n')[0].strip()
        env = MyEnvironment()
        obs = env.reset(seed=42, task_id='medium')
        action = parse_action(action_text, obs)
        if action is None:
            rewards.append(-1.0)
            continue
        next_obs = env.step(action)
        r = 0.3
        if env._checking >= 0: r += 0.3
        if 'ON TIME' in (next_obs.daily_summary or ''): r += 0.4
        if 'informal' in action_text: r -= 0.5
        rewards.append(max(-1.0, min(1.0, r)))
    return rewards


grpo_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=grpo_dataset,
    reward_funcs=reward_fn,
    args=GRPOConfig(
        output_dir='outputs/grpo',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=5e-5,
        logging_steps=5,
        max_completion_length=64,
        num_generations=4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        report_to='wandb' if os.environ.get('WANDB_API_KEY') else 'none',
    ),
)

print('🚀 GRPO training starting...')
grpo_stats = grpo_trainer.train()
model.save_pretrained('outputs/grpo')
tokenizer.save_pretrained('outputs/grpo')
print('✅ GRPO done!')

## Cell 8: Final Evaluation + Comparison

In [ ]:
FastLanguageModel.for_inference(model)

grpo_scores = {}
for task_id in ['easy', 'medium', 'hard']:
    scores = [run_trained_episode(model, tokenizer, task_id, seed=s) for s in range(5)]
    grpo_scores[task_id] = scores

print('\n' + '='*55)
print(f'{"Task":>8} | {"Heuristic":>10} | {"SFT (7B)":>10} | {"GRPO (7B)":>10}')
print('-'*55)
for t in ['easy','medium','hard']:
    h = sum(baseline_scores[t])/len(baseline_scores[t])
    s = sum(sft_scores[t])/len(sft_scores[t])
    g = sum(grpo_scores[t])/len(grpo_scores[t])
    print(f'{t:>8} | {h:>10.4f} | {s:>10.4f} | {g:>10.4f}')
print('='*55)

## Cell 9: Generate Reward Plots (save as .png)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json
import os

# Plot 1: Training loss
try:
    if 'trainer' in locals():
        sft_log = trainer.state.log_history
    else:
        with open('outputs/sft/trainer_state.json', 'r') as f:
            sft_log = json.load(f)['log_history']
            
    steps = [x['step'] for x in sft_log if 'loss' in x]
    losses = [x['loss'] for x in sft_log if 'loss' in x]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(steps, losses, color='#2196F3', linewidth=2)
    ax.set_xlabel('Training Step'); ax.set_ylabel('Loss')
    ax.set_title('SFT Training Loss (7B)'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig('training_loss.png', dpi=150)
    print('📈 Saved training_loss.png')
except Exception as e:
    print(f'⚠️ Could not generate SFT training loss plot: {e}')

# Plot 2: Before vs After
fig, ax = plt.subplots(figsize=(10,5))
tasks = ['Easy (30d)', 'Medium (60d)', 'Hard (90d)']
x = np.arange(3); w = 0.25
h_avg = [sum(baseline_scores[t])/len(baseline_scores[t]) for t in ['easy','medium','hard']]
s_avg = [sum(sft_scores[t])/len(sft_scores[t]) for t in ['easy','medium','hard']]
g_avg = [sum(grpo_scores[t])/len(grpo_scores[t]) for t in ['easy','medium','hard']]

ax.bar(x-w, h_avg, w, label='Heuristic', color='#FF9800')
ax.bar(x, s_avg, w, label='SFT (7B)', color='#2196F3')
ax.bar(x+w, g_avg, w, label='GRPO (7B)', color='#4CAF50')

for i in range(3):
    for v,dx in [(h_avg[i],-w),(s_avg[i],0),(g_avg[i],w)]:
        ax.text(i+dx, v+0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
        
ax.set_xticks(x); ax.set_xticklabels(tasks)
ax.set_ylabel('Episode Score (0-1)'); ax.set_ylim(0,1.1)
ax.set_title('Financial Triage — 7B Agent Performance'); ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('before_after_scores.png', dpi=150)
print('📊 Saved before_after_scores.png')
plt.show()


## ✅ Done!

**Files generated:**
- `training_loss.png` — SFT loss curve
- `before_after_scores.png` — Heuristic vs SFT vs GRPO
- `outputs/grpo/` — Trained LoRA adapters

**Next:** Download the `.png` files and add them to your README + HF Space.

**To train 3B instead:** In Cell 4, swap so the 3B `model_name` is active and the 7B line is commented, then re-run all cells.